# 情感分析：判断文本的正负面
> 难度标签：高级 | 预计时长：10-20分钟 | 前置知识：无学习经验


> 所属模块：08_自然语言处理 | 源文件：情感分析.py | 核心功能：情感词典、机器学习情感分类

## 概述

情感分析判断文本表达的情感倾向（正面/负面/中性）。应用场景：商品评论分析、舆情监控、客户反馈分析。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report
# --- 导入库 ---
from sklearn.pipeline import make_pipeline

## 数学原理

### 1. 情感分析的数学框架

情感分析是二分类问题：给定文本 $x$，预测情感标签 $y \in \{0, 1\}$（负面/正面）。

$$P(y=1|x) = \sigma(\mathbf{w}^\top \mathbf{x} + b)$$

其中 $\mathbf{x}$ 是文本的 TF-IDF 向量，$\sigma$ 是 sigmoid 函数。

### 2. TF-IDF 特征提取

将文本转换为 TF-IDF 向量：

$$\mathbf{x}_d = \text{TF-IDF}(d) = (\text{TF-IDF}(w_1, d), \ldots, \text{TF-IDF}(w_{|V|}, d))$$

代码中 `TfidfVectorizer(ngram_range=(1, 2))` 同时使用 unigram 和 bigram 特征，捕捉"不好"、"非常满意"等情感短语。

### 3. 逻辑回归分类器

逻辑回归模型：

$$P(y=1|\mathbf{x}) = \frac{1}{1 + e^{-(\mathbf{w}^\top \mathbf{x} + b)}}$$

**损失函数**（交叉熵）：

$$\mathcal{L} = -\frac{1}{n}\sum_{i=1}^{n}\left[y_i \log \hat{p}_i + (1-y_i)\log(1-\hat{p}_i)\right]$$

通过梯度下降优化 $\mathbf{w}, b$。

### 4. N-gram 的重要性

情感表达常依赖短语而非单个词：

| 表达 | unigram | bigram |
|------|---------|--------|
| "不好" | "好"（正面）| "不好"（负面）|
| "非常满意" | "满意" | "非常满意"（强正面）|
| "不推荐" | "推荐"（正面）| "不推荐"（负面）|

`ngram_range=(1, 2)` 同时保留 unigram 和 bigram，显著提升情感分类性能。

### 5. Pipeline 的数学流程

代码中 `make_pipeline(TfidfVectorizer(), LogisticRegression())` 定义的完整流程：

$$\text{文本} \xrightarrow{\text{TF-IDF}} \mathbf{x} \xrightarrow{\text{LR}} P(y=1|\mathbf{x}) \xrightarrow{\text{threshold}} \hat{y}$$

### 6. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `TfidfVectorizer(ngram_range=(1,2))` | 构建 TF-IDF 特征矩阵 $\hat{X}$ |
| `LogisticRegression(max_iter=1000)` | 逻辑回归：$P(y=1|x) = \sigma(w^Tx+b)$ |
| `cross_val_score` | K 折交叉验证评估泛化能力 |
| `classification_report` | 精确率、召回率、F1 值 |
| `make_pipeline()` | 端到端流程：向量化 $\to$ 分类 |

### 1. 构造中文情感分析数据

运行 1. 构造中文情感分析数据 的代码，观察算法在该环节的行为。

In [ ]:
positive = [
    "这部电影非常精彩，值得一看",
    "服务态度很好，环境优美",
    "产品性价比很高，推荐购买",
    "味道不错，分量也足",
# --- 继续 ---
    "体验非常好，下次还来",
    "内容丰富，讲解清晰",
    "物流很快，包装完好",
    "价格实惠，质量不错",
    "非常满意，五星好评",
# --- 继续 ---
    "剧情紧凑，演员演技在线",
    "酒店位置好，房间干净整洁",
    "课程实用，学到很多知识",
] * 5  # 增加样本数

negative = [
    "太失望了，完全不值这个价",
    "服务态度很差，再也不来了",
    "质量很差，用了一天就坏了",
    "味道一般，分量太少",
# --- 继续 ---
    "体验很差，浪费时间",
    "内容空洞，没什么干货",
    "物流太慢，等了一个星期",
    "价格太贵，不如买别家的",
    "非常失望，差评",
# --- 继续 ---
    "剧情拖沓，演技尴尬",
    "房间很脏，设施老旧",
    "课程太浅，学不到东西",
] * 5

texts = positive + negative
labels = [1] * len(positive) + [0] * len(negative)

### 2. 文本向量化 + 分类

在分类任务上训练模型并评估性能。

In [ ]:
# 用 TF-IDF 提取特征
model = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2)),
    LogisticRegression(max_iter=1000),
)

# 交叉验证
scores = cross_val_score(model, texts, labels, cv=5, scoring="accuracy")
print("=== 情感分析（TF-IDF + 逻辑回归）===")
print(f"5 折交叉验证准确率: {scores.mean():.4f} ± {scores.std():.4f}")

### 3. 训练和预测

使用训练好的模型进行预测，观察输出结果。

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(texts, labels, test_size=0.2, random_state=42)
model.fit(X_train, y_train)

print(f"\n训练集大小: {len(X_train)}, 测试集大小: {len(X_test)}")
y_pred = model.predict(X_test)
print(f"测试准确率: {model.score(X_test, y_test):.4f}")

### 4. 预测新文本

使用训练好的模型进行预测，观察输出结果。

In [ ]:
print("\n=== 预测新文本 ===")
new_texts = [
    "这个产品真的太好了，强烈推荐！",
    "质量太差了，完全是骗钱的",
    "还行吧，一般般",
# --- 继续 ---
    "服务热情，环境不错，下次还来",
]
probs = model.predict_proba(new_texts)
preds = model.predict(new_texts)
for text, pred, prob in zip(new_texts, preds, probs):
# --- 条件判断 ---
    sentiment = "正面" if pred == 1 else "负面"
    print(f"  [{sentiment}] {text} (置信度: {max(prob):.3f})")

### 5. 特征词分析

运行 5. 特征词分析 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 影响情感判断的关键词 ===")
tfidf = model.named_steps["tfidfvectorizer"]
lr = model.named_steps["logisticregression"]
feature_names = tfidf.get_feature_names_out()
coefs = lr.coef_[0]

# 正面词
top_pos = np.argsort(coefs)[::-1][:10]
print("正面情感关键词:")
for idx in top_pos:
    print(f"  {feature_names[idx]}: {coefs[idx]:.4f}")

# 负面词
top_neg = np.argsort(coefs)[:10]
print("负面情感关键词:")
for idx in top_neg:
    print(f"  {feature_names[idx]}: {coefs[idx]:.4f}")

### 6. 不同模型对比

对比不同模型或配置的性能差异。

In [ ]:
print("\n=== 不同分类器对比 ===")
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

for name, clf in [("LogisticRegression", LogisticRegression(max_iter=1000)),
                   ("MultinomialNB", MultinomialNB()),
                   ("LinearSVC", LinearSVC(max_iter=1000)),
                   ("RandomForest", RandomForestClassifier(n_estimators=50))]:
    pipe = make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), clf)
# --- 继续 ---
    scores = cross_val_score(pipe, texts, labels, cv=5, scoring="accuracy")
    print(f"  {name:>20}: {scores.mean():.4f} ± {scores.std():.4f}")

print("\n=== 情感分析要点 ===")
print("- 中文需要先分词（jieba），英文可直接用空格分割")
print("- TF-IDF + 简单分类器是很好的 baseline")
print("- N-gram (1,2) 可以捕捉短语信息（如'不太好'）")
print("- 预训练模型（BERT）效果更好但需要 GPU")
# --- 输出结果 ---
print("- 数据质量是关键：标注一致性比数据量更重要")

## 关键代码解释

两种方法：基于情感词典（规则）和基于机器学习（数据驱动）。

## 注意事项

1. 讽刺和反语是情感分析的难题
2. 领域特定情感词很重要
3. 细粒度情感分析（方面级）比文档级更有价值

## 总结与延伸

以上代码展示了 **情感分析** 的完整流程。

**解读要点**：
- 观察 **词汇表大小** 和 **向量维度** 是否合理
- 检查相似词查询结果是否符合语义直觉
- 关注分类任务中的 **TF-IDF 权重** 分布

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **方面级情感分析**：针对产品的不同方面分别判断情感
- **多模态情感分析**：结合文本、语音、表情
- **ChatGPT 的情感理解能力**：大模型是否需要专门的情感分析模型
